# Silverwing-ML - Production Training on Kaggle (Free GPU)

Twin of `colab_production.ipynb`: same 102M model, same data, same hyperparams - so
**checkpoints are interchangeable** between Colab and Kaggle. Persistence uses a private
Kaggle dataset (`<USER>/silverwing-state`) instead of Drive, synced automatically every ~15 min
while training runs, so a killed session loses at most ~15 min of work.

**One-time setup:**
1. Verify your phone at kaggle.com (required to enable Internet).
2. Seed your existing Colab/Drive progress: run `colab_drive_to_kaggle.ipynb` once on Colab.
3. Import this notebook: kaggle.com/code > New Notebook > File > Import Notebook.
4. Sidebar > Session options: **Accelerator = GPU T4 x2 or P100**, **Internet = On**.

**Budget:** 30 h GPU/week, sessions up to ~12 h. At ~8-10K tok/s fp16 the full 36k-step
pretrain plus SFT fits comfortably inside a week even sharing hours with Colab.

**Every session:** run cells 1-7 in order. Cell 7 resumes from the newest checkpoint
automatically; just re-run it if anything dies.

In [ ]:
# Cell 1: Kaggle identity + state sync helpers (replaces Google Drive)
# All progress lives in ONE private dataset <USER>/silverwing-state:
#   checkpoints/pretrain/, checkpoints/sft/
#   corpus/corpus-external/, tokenizer/tokenizer-v2/
import json

import shutil

import subprocess

from pathlib import Path

USER = 'videlisndichi'  # <-- EDIT ME
STATE = f'{USER}/silverwing-state'
WORK = Path('/kaggle/working')
STATE_DIR = WORK / 'state'
STAGE = WORK / '.stage'


def _kaggle(*args):
    return subprocess.run(['kaggle', *args], capture_output=True, text=True)


def dataset_exists():
    return _kaggle('datasets', 'status', STATE).returncode == 0


def pull_state():
    """Download newest dataset version into STATE_DIR. False = fresh start."""
    if not dataset_exists():
        print(f'{STATE} does not exist yet - fresh start')
        return False
    dl = WORK / '.dl'
    if dl.exists():
        shutil.rmtree(dl)
    dl.mkdir(parents=True)
    r = _kaggle('datasets', 'download', STATE, '-p', str(dl), '--unzip')
    assert r.returncode == 0, f'download failed:\n{r.stderr[-1500:]}'
    if STATE_DIR.exists():
        shutil.rmtree(STATE_DIR)
    shutil.move(str(dl), STATE_DIR)
    freed = 0
    tc_ok = _kaggle('datasets', 'status', f'{USER}/silverwing-tokcache').returncode == 0
    for f in (STATE_DIR / 'corpus/corpus-external').glob('.token-cache-*.pt'):
        if tc_ok:
            freed += f.stat().st_size
            f.unlink()
    if freed:
        print(f'dropped {freed / 1e9:.1f} GB token caches from local state '
              '(they are fetched separately from silverwing-tokcache)')
    ck = STATE_DIR / 'checkpoints/pretrain'
    n = len(list(ck.glob('step-*.pt'))) if ck.exists() else 0
    print(f'State restored from {STATE} ({n} pretrain step checkpoints)')
    return True


def _prune_steps(root, keep):
    """Delete oldest step-*.pt under root, keeping the newest `keep` (best.pt untouched)."""
    for name in ['checkpoints/pretrain', 'checkpoints/sft']:
        ck = root / name
        if not ck.exists():
            continue
        steps = sorted(ck.glob('step-*.pt'),
                       key=lambda p: int(p.stem.split('-')[1]))
        for p in steps[:-keep]:
            p.unlink()


def push_state(msg, keep_last_steps=2):
    """Push live state as a dataset version without filling /kaggle/working:
    stale stage mirrors are wiped up front, old step checkpoints are pruned
    in BOTH the live state and the staged copy, the stage is deleted after
    the push, and failures warn instead of killing training."""
    if STAGE.exists():
        shutil.rmtree(STAGE)
    _prune_steps(STATE_DIR, keep_last_steps)
    ok, detail = False, ''
    try:
        for sub in ['checkpoints/pretrain', 'checkpoints/sft',
                    'corpus/corpus-external', 'tokenizer/tokenizer-v2']:
            src = STATE_DIR / sub
            if not src.exists():
                continue
            dst = STAGE / sub
            dst.mkdir(parents=True, exist_ok=True)
            for f in src.iterdir():
                if f.is_file() and not f.name.startswith('.'):
                    shutil.copy2(f, dst / f.name)
        _prune_steps(STAGE, keep_last_steps)
        meta = {'title': 'Silverwing State', 'id': STATE,
                'licenses': [{'name': 'CC0-1.0'}]}
        (STAGE / 'dataset-metadata.json').write_text(json.dumps(meta))
        verb = 'version' if dataset_exists() else 'create'
        args = ['datasets', verb, '-p', str(STAGE), '--dir-mode', 'zip']
        if verb == 'version':
            args += ['-m', msg]
        r = _kaggle(*args)
        ok = r.returncode == 0
        if not ok:
            detail = f'{r.stdout[-600:]}\n{r.stderr[-600:]}'
    except OSError as e:
        detail = repr(e)
    size = sum(f.stat().st_size for f in STAGE.rglob('*') if f.is_file()) / 1e9
    shutil.rmtree(STAGE, ignore_errors=True)
    free = shutil.disk_usage(WORK).free / 1e9
    if ok:
        print(f'State pushed ({size:.2f} GB staged, {free:.1f} GB free): {msg}')
    else:
        print(f'[warn] state push failed ({msg}): {detail} ({free:.1f} GB free)')


pull_state()

In [ ]:
# Cell 2: Dependencies + compute probe (torch ships preinstalled on Kaggle).
# Fails fast with a clear message if the assigned GPU is unsupported.
import os

import sys

import warnings

os.environ.setdefault('PYDEVD_DISABLE_FILE_VALIDATION', '1')
warnings.filterwarnings('ignore', message='.*CUDA capability.*')
warnings.filterwarnings('ignore', message='.*Found GPU.*')
warnings.filterwarnings('ignore', message='.*Please install PyTorch.*')

!pip install -q pyyaml numpy datasets huggingface_hub zstandard

import datasets
import torch
import zstandard

print(f'torch {torch.__version__} | datasets {datasets.__version__} | '
      f'zstandard {zstandard.__version__}')
if not torch.cuda.is_available():
    sys.exit('NO GPU - Session options > Accelerator > GPU T4 x2, then rerun')
arch_list = torch.cuda.get_arch_list()
print('torch CUDA archs:', ' '.join(arch_list))
for i in range(torch.cuda.device_count()):
    p = torch.cuda.get_device_properties(i)
    cap = f'sm_{p.major}{p.minor}'
    ok = cap in arch_list or f'compute_{p.major}{p.minor}' in arch_list
    status = 'OK' if ok else 'UNSUPPORTED BY THIS TORCH BUILD'
    print(f'GPU {i}: {p.name} ({p.total_memory / 1e9:.0f} GB, {cap}) -> {status}')
    if not ok:
        sys.exit(
            f'{p.name} ({cap}) cannot run this torch build. '
            'Session options > Accelerator > GPU T4 x2, then rerun.'
        )

In [ ]:
# Cell 3: Clone the repo and HARD-SYNC to latest origin/main
# Run cells strictly in order within a session (clone wipes experiments/)
import os

REPO = '/kaggle/working/Silverwing-ML'
if not os.path.exists(REPO):
    !git clone https://github.com/oledesug-source/silverwing-ml.git {REPO}
os.chdir(REPO)
!git fetch origin
!git reset --hard origin/main
!git clean -fdq
head = !git rev-parse --short HEAD
print('repo at commit:', head[0])

In [ ]:
# Cell 4: Corpus - restore from state, else download+process once and push
import shutil
import subprocess
import sys
from pathlib import Path

CORPUS = Path('experiments/corpus-external')
RAW = Path('experiments/ingested')
S_CORPUS = STATE_DIR / 'corpus/corpus-external'

assert Path('scripts/ingest_external_v2.py').exists(), 'repo sync failed - rerun Cell 3'


def _run(cmd):
    print('Running:', ' '.join(cmd))
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True)
    for line in proc.stdout:
        print(line, end='')
    return proc.wait()


if S_CORPUS.exists() and any(S_CORPUS.glob('train.*.jsonl')):
    if CORPUS.exists():
        shutil.rmtree(CORPUS)
    shutil.copytree(S_CORPUS, CORPUS)
    print('Corpus restored:', sorted(p.name for p in CORPUS.glob('*.jsonl')))
else:
    rc = _run([sys.executable, 'scripts/ingest_external_v2.py',
               'download', '--preset', 'all', '--skip-existing'])
    assert rc == 0, f'download failed (exit {rc}) - just rerun this cell'
    rc = _run([sys.executable, 'scripts/ingest_external_v2.py',
               'process', '--input', str(RAW / 'all')])
    assert rc == 0, f'processing failed (exit {rc}) - rerun will NOT re-download'
    shutil.copytree(CORPUS, S_CORPUS, dirs_exist_ok=True)
    push_state('corpus built')

TCACHE_DS = f'{USER}/silverwing-tokcache'
r = _kaggle('datasets', 'download', TCACHE_DS, '-p', str(CORPUS), '--unzip')
if r.returncode == 0:
    print('token caches fetched - tokenization skipped')
else:
    print('[warn] no tokcache yet - trainer will tokenize once, then cache locally')

train_shard = next(CORPUS.glob('train.*.jsonl'))
print(f"train documents: {sum(1 for _ in open(train_shard, encoding='utf-8')):,}")

In [ ]:
# Cell 5: Tokenizer v2 - MUST be the same tokenizer your checkpoints use
import shutil
import subprocess
import sys
from pathlib import Path

TOK = Path('experiments/tokenizer-v2')
S_TOK = STATE_DIR / 'tokenizer/tokenizer-v2'

if S_TOK.exists() and (S_TOK / 'vocab.json').exists():
    if TOK.exists():
        shutil.rmtree(TOK)
    shutil.copytree(S_TOK, TOK)
    print('Tokenizer v2 restored from state')
else:
    print('[warn] no tokenizer in state - training a fresh one.')
    print('[warn] If resuming Colab checkpoints that used the committed tokenizer')
    print('[warn] instead, set TOKENIZER_DIR = experiments/tokenizer manually.')
    r = subprocess.run(
        [sys.executable, 'scripts/train_tokenizer.py',
         '--corpus-dir', 'experiments/corpus-external',
         '--vocab-size', '16384',
         '--max-documents', '60000',
         '--output-dir', 'experiments/tokenizer-v2'],
        capture_output=True, text=True)
    print(r.stdout[-1200:], r.stderr[-1200:])
    assert r.returncode == 0, 'tokenizer training failed'
    shutil.copytree(TOK, S_TOK, dirs_exist_ok=True)

TOKENIZER_DIR = 'experiments/tokenizer-v2'
print('TOKENIZER_DIR =', TOKENIZER_DIR)

In [ ]:
# Cell 6: Production pretraining config - values MUST match your Colab run.
# The trainer hashes these into every checkpoint (resume guard); only
# checkpoint_dir may differ between platforms.
import yaml

cfg = {'training': {
    'version': 'training-v1',
    'model_config_path': 'configs/model.yaml',
    'corpus_dir': 'experiments/corpus-external',
    'tokenizer_dir': TOKENIZER_DIR,
    'checkpoint_dir': f'{STATE_DIR}/checkpoints/pretrain',
    'batch_size': 16,
    'grad_accum_steps': 1,
    'block_size': 512,
    'max_steps': 36000,
    'warmup_steps': 2000,
    'lr': 3.0e-4,
    'min_lr_ratio': 0.1,
    'weight_decay': 0.1,
    'betas': [0.9, 0.95],
    'eps': 1.0e-8,
    'grad_clip': 1.0,
    'seed': 42,
    'log_steps': 50,
    'eval_steps': 1000,
    'eval_sequences': 16,
    'save_steps': 1000,
    'verify_dataset': False,
    'expected_dataset_hash': None,
    'require_validation': True,
    'require_clean_repo': False,
    'device': 'cpu',
    'amp': True,
    'amp_dtype': 'float16',
}}
with open('configs/training_production.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)
t = cfg['training']
print('configs/training_production.yaml written')
print(f"target: {t['max_steps']:,} steps x {t['batch_size'] * t['block_size']:,} tok/step")

## Pretrain
Output: `<state>/checkpoints/pretrain/best.pt` + step checkpoints, auto-pushed to the dataset.

In [ ]:
# Cell 7: PRETRAIN RUNNER - safe to re-run any number of times.
# Resumes from the newest local checkpoint; syncs state every ~15 min.
import os
import re
import subprocess
import sys
import threading
import time
from pathlib import Path

CKPT = STATE_DIR / 'checkpoints/pretrain'
TARGET = 36000
SYNC_EVERY_MIN = 15
# Resumed runs used to die on an RNG-state bug (fixed on main cde7ef8);
# resume is safe again. Flip to True only for a deliberate clean start.
FRESH_START = False


def latest_step(d):
    steps = []
    for p in d.glob('step-*.pt'):
        m = re.match(r'step-(\d+)\.pt', p.name)
        if m:
            steps.append(int(m.group(1)))
    return max(steps) if steps else 0


done = latest_step(CKPT)
if done >= TARGET:
    print(f'PRETRAIN COMPLETE ({done:,} steps). Continue to Cell 8.')
else:
    cks = sorted(CKPT.glob('step-*.pt'),
                 key=lambda p: int(re.sub(r'\D', '', p.name)))
    if FRESH_START and cks:
        for p in CKPT.glob('step-*.pt'):
            p.unlink()
        for nm in ('best.pt', 'final.pt'):
            q = CKPT / nm
            if q.exists():
                q.unlink()
        print(f'FRESH START - deleted {len(cks)} stale local checkpoint(s) '
              f'(+best/final) to free disk; originals stay in the dataset')
        done = 0
    cmd = [sys.executable, 'scripts/train.py',
           '--config', 'configs/training_production.yaml',
           '--device', 'cuda',
           '--no-clean-repo-check']
    if cks and not FRESH_START:
        cmd += ['--resume-from', str(cks[-1])]
        print(f'Resuming from step {done:,} / {TARGET:,}')
    else:
        if cks:
            print(f'FRESH START - ignoring {len(cks)} local checkpoint(s) '
                  f'(set FRESH_START = False to resume)')
        print('Fresh pretraining start')
    t0 = time.time()
    proc = subprocess.Popen(cmd, env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    last_pushed, last_t, polls = done, 0.0, 0

    def _drop_local_caches():
        """Free ~4 GB once the trainer has loaded tokens into RAM (they are
        re-fetched from silverwing-tokcache next session)."""
        time.sleep(420)
        freed = 0
        for d in (Path('experiments/corpus-external'),
                  STATE_DIR / 'corpus/corpus-external'):
            if d.exists():
                for f in d.glob('.token-cache-*.pt'):
                    freed += f.stat().st_size
                    f.unlink()
        print(f'[watchdog] dropped token caches from disk '
              f'({freed / 1e9:.1f} GB freed; copies stay in the dataset)', flush=True)

    threading.Thread(target=_drop_local_caches, daemon=True).start()
    try:
        while proc.poll() is None:
            time.sleep(30)
            polls += 1
            cur = latest_step(CKPT)
            if polls % 10 == 0:
                print(f'[watchdog] trainer alive {int((time.time() - t0) / 60)} min, '
                      f'latest checkpoint step {cur:,}', flush=True)
            if cur > last_pushed and time.time() - last_t > SYNC_EVERY_MIN * 60:
                push_state(f'pretrain step {cur}')
                last_pushed, last_t = cur, time.time()
    except KeyboardInterrupt:
        proc.terminate()
        print('Interrupted - terminating trainer...')
        time.sleep(10)
    finally:
        rc = proc.wait()
        cur = latest_step(CKPT)
        if cur > 0:
            push_state(f'pretrain step {cur} (session end)')
    print(f'exit={rc} elapsed={(time.time() - t0) / 60:.1f} min')
    print(f'progress: {latest_step(CKPT):,} / {TARGET:,} steps')
    if latest_step(CKPT) < TARGET:
        print('Session ended early? Run this cell again to continue.')

In [ ]:
# Cell 8: Verify SFT dataset (committed in repo)
from pathlib import Path

sft_ds = Path('experiments/sft/sft-v2-all.jsonl')
assert sft_ds.exists(), 'SFT dataset missing from repo - check git clone'
n = sum(1 for _ in open(sft_ds, encoding='utf-8'))
print(f'SFT examples: {n:,}')

In [ ]:
# Cell 9: SFT run (fp16 AMP, init from pretrained best, state-synced).
# Skips itself when pretraining has not reached 36,000 steps yet.
import os
import re
import subprocess
import sys
from pathlib import Path

import yaml

_pre_ckpts = Path(f'{STATE_DIR}/checkpoints/pretrain')
PRETRAIN_DONE = max([int(m.group(1)) for p in _pre_ckpts.glob('step-*.pt')
                     if (m := re.match(r'step-(\d+)\.pt', p.name))] or [0])
PRETRAIN_BEST = f'{STATE_DIR}/checkpoints/pretrain/best.pt'
print(f'pretrain progress: {PRETRAIN_DONE:,} / 36,000 steps')

cfg = {'sft': {
    'version': 'sft-v1',
    'model_config_path': 'configs/model.yaml',
    'tokenizer_dir': TOKENIZER_DIR,
    'init_from': PRETRAIN_BEST,
    'dataset_path': 'experiments/sft/sft-v2-all.jsonl',
    'checkpoint_dir': f'{STATE_DIR}/checkpoints/sft',
    'batch_size': 16,
    'block_size': 512,
    'max_steps': 2000,
    'warmup_steps': 100,
    'lr': 1.0e-4,
    'min_lr_ratio': 0.1,
    'weight_decay': 0.1,
    'betas': [0.9, 0.95],
    'eps': 1.0e-8,
    'grad_clip': 1.0,
    'seed': 42,
    'log_steps': 25,
    'eval_steps': 200,
    'eval_examples': 64,
    'save_steps': 500,
    'eval_fraction': 0.05,
    'require_clean_repo': False,
    'device': 'cpu',
    'amp': True,
    'amp_dtype': 'float16',
}}
with open('configs/sft_production.yaml', 'w') as f:
    yaml.safe_dump(cfg, f)

if PRETRAIN_DONE >= 36000 and os.path.exists(PRETRAIN_BEST):
    r = subprocess.run([sys.executable, 'scripts/train_sft.py',
                        '--config', 'configs/sft_production.yaml',
                        '--device', 'cuda',
                        '--no-clean-repo-check'],
                       env={**os.environ, 'PYTHONUNBUFFERED': '1'})
    assert r.returncode == 0, 'SFT run failed'
    push_state('sft complete')
else:
    print('Skipping SFT: pretrain not finished. Rerun Cell 7, then this cell.')

In [ ]:
# Cell 10: Generation test (skips itself until SFT has run)
import os

import sys

import torch

if not os.path.exists(f'{STATE_DIR}/checkpoints/sft/best.pt'):
    print('SFT best.pt not ready - skipping generation test. Run Cell 9 after pretraining completes.')
else:
    sys.path.insert(0, '.')
    from foundation.inference import Generator
    from foundation.model.config import ModelConfig
    from foundation.model.model import SilverwingDecoder
    from foundation.tokenizer import TokenizerV2

    tok = TokenizerV2.load(TOKENIZER_DIR)
    ckpt_path = f'{STATE_DIR}/checkpoints/sft/best.pt'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model_cfg = ModelConfig.from_yaml('configs/model.yaml')
    model = SilverwingDecoder(model_cfg)
    state = torch.load(ckpt_path, map_location=device, weights_only=False)
    model.load_state_dict(state.get('model_state', state))
    model = model.to(device).eval()
    print(f'Model: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M params')

    gen = Generator(model, tok)
    prompts = [
        'What is 2 + 2? ',
        'The answer to 3 * 3 is ',
        'Solve for x: 2x + 5 = 13. Step 1:',
        'The capital of France is ',
    ]
    for p in prompts:
        result = gen.generate(p, max_new_tokens=80, temperature=0.7, top_k=50)
        print(f'--- {p!r}\n{result}')

In [ ]:
# Cell 11: Math benchmark against regression gates (skips until SFT has run)
import os

import sys
import subprocess

if os.path.exists(f'{STATE_DIR}/checkpoints/sft/best.pt'):
    subprocess.run([sys.executable, 'scripts/run_benchmark.py',
                    '--benchmark', 'math-benchmark-v1',
                    '--model', f'silverwing:{STATE_DIR}/checkpoints/sft/best.pt',
                    '--tokenizer-dir', TOKENIZER_DIR])
else:
    print('SFT best.pt not ready - skipping benchmark. Run Cell 9 after pretraining completes.')

In [ ]:
# Cell 12: Summary + final state push
import json
from pathlib import Path

for name, report in [('pretrain', STATE_DIR / 'checkpoints/pretrain/training_report.json'),
                     ('sft', STATE_DIR / 'checkpoints/sft/sft_report.json')]:
    if report.exists():
        d = json.loads(report.read_text(encoding='utf-8'))
        print(f'[{name}] eval_loss={d.get("final_eval_loss")} ppl={d.get("final_perplexity")} '
              f'best={d.get("best_eval_loss")}')
    else:
        print(f'[{name}] report not found yet')
push_state('final artifacts')
print('\nTo continue on Colab instead, run colab_drive_to_kaggle.ipynb Cell 4 (pull).')